In [1]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [2]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl

Processing /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl


# Reproducibility

In [3]:
# Reproducibility
import torch, random, numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Paths

In [4]:
TRAIN_SEQ = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv"

VAL_SEQ   = "/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_sequences.csv"
VAL_LBL   = "/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_labels.csv"

TEST_SEQ  = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"


# Dataset

In [5]:
import pandas as pd
from torch.utils.data import Dataset
from Bio.Seq import Seq
import math
import torch

NUC_MAP = {'A':0,'U':1,'G':2,'C':3}


def clean_sequence(seq):
    seq = str(Seq(seq.upper()))
    return "".join([s for s in seq if s in NUC_MAP])


def one_hot(seq):
    x = torch.zeros(len(seq),4)
    for i,s in enumerate(seq):
        x[i,NUC_MAP[s]] = 1
    return x


# ---------- sinusoidal encoding ----------
def positional_encoding(L, dim=8):
    pos = torch.arange(L).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, dim, 2) * (-math.log(10000.0)/dim)
    )

    pe = torch.zeros(L, dim)
    pe[:,0::2] = torch.sin(pos*div)
    pe[:,1::2] = torch.cos(pos*div)
    return pe


class RNADataset(Dataset):

    def __init__(self, seq_csv, label_csv=None, max_length=None):
        """
        max_length:
            int  -> truncate (training/validation)
            None -> FULL sequence (test/inference)
        """

        self.q_df = pd.read_csv(seq_csv)
        self.max_length = max_length
        self.has_labels = label_csv is not None

        if self.has_labels:

            labels = pd.read_csv(label_csv, low_memory=False)
            labels["struct_id"] = labels["ID"].str.split("_").str[0]
            labels["res_idx"]   = labels["ID"].str.split("_").str[1].astype(int)

            self.structures = {}

            for sid, g in labels.groupby("struct_id"):

                g = g.sort_values("res_idx")

                coords = torch.tensor(
                    g[["x_1","y_1","z_1"]].values,
                    dtype=torch.float32
                )

                # remove NaN residues only (structure safe)
                mask = ~torch.isnan(coords).any(dim=1)
                coords = coords[mask]

                if len(coords) > 0:
                    self.structures[sid] = coords

            self.valid_ids = [
                sid for sid in self.q_df.target_id
                if sid in self.structures
            ]

        else:
            self.valid_ids = list(self.q_df.target_id)

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):

        sid = self.valid_ids[idx]
        row = self.q_df[self.q_df.target_id == sid].iloc[0]

        seq = clean_sequence(row.sequence)

        coords = None
        if self.has_labels:

            coords = self.structures[sid]

            # ensure seq & coords same length
            L = min(len(seq), coords.shape[0])
            seq = seq[:L]
            coords = coords[:L]

            # normalization (translation + scale invariant)
            coords = coords - coords.mean(0, keepdim=True)
            coords = coords / (coords.std() + 1e-8)

        # ✅ SAFE TRUNCATION (ONLY IF ENABLED)
        if self.max_length is not None and len(seq) > self.max_length:
            seq = seq[:self.max_length]
            if coords is not None:
                coords = coords[:self.max_length]

        # features
        x = one_hot(seq)
        pos_enc = positional_encoding(len(seq), 8)
        x = torch.cat([x, pos_enc], dim=1)

        return sid, x, coords

# Initialize Dataset

In [6]:
train_dataset = RNADataset(TRAIN_SEQ, TRAIN_LBL, max_length=1000)
val_dataset   = RNADataset(VAL_SEQ, VAL_LBL, max_length=1000)

# ⭐ FULL RNA — NO TRUNCATION
test_dataset  = RNADataset(TEST_SEQ, None, max_length=None)

# Graph Builder

In [6]:
from torch_geometric.data import Data
import torch

def center_coordinates(coords):
    return coords - coords.mean(0, keepdim=True)

def build_graph(x, coords=None, k=16):

    L = x.size(0)
    edges = []
    attrs = []

    if coords is not None:
        coords = center_coordinates(coords)

    for i in range(L):
        for j in range(max(0, i - k), min(L, i + k + 1)):
            if i == j:
                continue

            edges.append([i, j])

            seq_gap = float(abs(i - j))

            if coords is not None:
                d = torch.norm(coords[i] - coords[j]).item()
                # 2-D edge feature
                attrs.append([d, seq_gap])
            else:
                # test/inference time (no coords)
                attrs.append([seq_gap, 0.0])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(attrs, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    if coords is not None:
        data.pos = coords
        data.y   = coords

    return data

# Build Graph Sets

In [7]:
train_dataset = RNADataset(TRAIN_SEQ, TRAIN_LBL)
val_dataset   = RNADataset(VAL_SEQ, VAL_LBL)
test_dataset  = RNADataset(TEST_SEQ, None)

train_graphs=[build_graph(x,c) for _,x,c in train_dataset]
val_graphs  =[build_graph(x,c) for _,x,c in val_dataset]

test_graphs=[]
for _,x,_ in test_dataset:
    g=build_graph(x,None)
    g.pos=torch.zeros(g.num_nodes,3)
    test_graphs.append(g)

# Dataloaders

In [8]:
train_loader = DataLoader(train_graphs, batch_size=4, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(val_graphs,   batch_size=4, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_graphs,  batch_size=4, shuffle=False, pin_memory=True)

# EGNN
- deeper network

- residual updates

- stable coordinate scaling

In [9]:
import torch.nn as nn
from torch_geometric.utils import scatter

class EGNNLayer(nn.Module):

    def __init__(self, hidden, edge_dim=2):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden*2+edge_dim+1,hidden),
            nn.SiLU(),
            nn.Linear(hidden,hidden)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden*2,hidden),
            nn.SiLU(),
            nn.Linear(hidden,hidden)
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden,1),
            nn.Tanh()
        )

    def forward(self,x,pos,edge_index,edge_attr):

        row,col=edge_index

        rel=pos[row]-pos[col]
        dist2=(rel**2).sum(1,keepdim=True)

        m=self.edge_mlp(
            torch.cat([x[row],x[col],edge_attr,dist2],1)
        )

        agg=scatter(m,row,dim=0,dim_size=x.size(0),reduce="mean")

        x=x+self.node_mlp(torch.cat([x,agg],1))

        trans=self.coord_mlp(m)*rel
        delta=scatter(trans,row,dim=0,dim_size=pos.size(0),reduce="mean")

        pos=pos+0.1*delta   # stability scale

        return x,pos

### Full Model

In [10]:
class EGNNModel(nn.Module):

    def __init__(self,in_dim=12,hidden=128,layers=6):
        super().__init__()

        self.embed=nn.Linear(in_dim,hidden)

        self.layers=nn.ModuleList([
            EGNNLayer(hidden) for _ in range(layers)
        ])

        self.out=nn.Linear(hidden,3)

    def forward(self,data):

        x=self.embed(data.x)
        pos=data.pos

        for layer in self.layers:
            x,pos=layer(x,pos,data.edge_index,data.edge_attr)

        return self.out(x)

# Training Utilities

In [11]:
model=EGNNModel().to(device)

optimizer=torch.optim.Adam(model.parameters(),lr=3e-4)
loss_fn=nn.MSELoss()

def mae(pred,target):
    return (pred-target).abs().mean().item()

# Train / Validate

In [12]:
def train_epoch():
    model.train()
    tl,tm=0,0

    for batch in train_loader:
        batch=batch.to(device)

        optimizer.zero_grad()
        pred=model(batch)
        loss=loss_fn(pred,batch.y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()

        tl+=loss.item()
        tm+=mae(pred,batch.y)

    return tl/len(train_loader),tm/len(train_loader)


@torch.no_grad()
def validate():
    model.eval()
    vl,vm=0,0

    for batch in val_loader:
        batch=batch.to(device)
        pred=model(batch)
        vl+=loss_fn(pred,batch.y).item()
        vm+=mae(pred,batch.y)

    return vl/len(val_loader),vm/len(val_loader)

### Training Loop

In [13]:
EPOCHS=40

for e in range(EPOCHS):

    tr_loss,tr_mae=train_epoch()
    va_loss,va_mae=validate()

    print(
        f"Epoch {e} | "
        f"Train MSE {tr_loss:.4f} | "
        f"Val MSE {va_loss:.4f}"
    )

Epoch 0 | Train MSE 0.7901 | Val MSE 0.9960
Epoch 1 | Train MSE 0.7881 | Val MSE 0.9976
Epoch 2 | Train MSE 0.7872 | Val MSE 1.0026
Epoch 3 | Train MSE 0.7907 | Val MSE 1.0049
Epoch 4 | Train MSE 0.7903 | Val MSE 1.0009
Epoch 5 | Train MSE 0.7907 | Val MSE 1.0084
Epoch 10 | Train MSE 0.7875 | Val MSE 1.0018
Epoch 11 | Train MSE 0.7869 | Val MSE 1.0077
Epoch 12 | Train MSE 0.7847 | Val MSE 1.0163
Epoch 13 | Train MSE 0.7871 | Val MSE 0.9851
Epoch 14 | Train MSE 0.7921 | Val MSE 1.0014
Epoch 15 | Train MSE 0.7897 | Val MSE 1.0140
Epoch 16 | Train MSE 0.7872 | Val MSE 1.0008
Epoch 17 | Train MSE 0.7870 | Val MSE 1.0340
Epoch 18 | Train MSE 0.7839 | Val MSE 1.0274
Epoch 19 | Train MSE 0.7827 | Val MSE 1.0442
Epoch 20 | Train MSE 0.7830 | Val MSE 1.0221
Epoch 21 | Train MSE 0.7820 | Val MSE 1.0905
Epoch 22 | Train MSE 0.7831 | Val MSE 1.0613
Epoch 23 | Train MSE 0.7804 | Val MSE 0.9992
Epoch 24 | Train MSE 0.7792 | Val MSE 1.0335
Epoch 25 | Train MSE 0.7767 | Val MSE 1.0455
Epoch 26 | Train

# Test Inference

In [14]:
@torch.no_grad()
def run_test():
    model.eval()
    preds=[]
    for batch in test_loader:
        batch=batch.to(device)
        preds.append(model(batch).cpu())
    return torch.cat(preds)

test_predictions=run_test()
print(test_predictions.shape)

torch.Size([5662, 3])


# Submission Builder

In [15]:
IDX2NUC={0:'A',1:'U',2:'G',3:'C'}

def onehot_to_base(x):
    return IDX2NUC[int(x[:4].argmax())]

rows=[]
ptr=0

for sid,x,_ in test_dataset:

    L=x.shape[0]
    coords=test_predictions[ptr:ptr+L]
    ptr+=L

    for i in range(L):

        rows.append({
            "ID":f"{sid}_{i+1}",
            "resname":onehot_to_base(x[i]),
            "resid":i+1,
            "x_1":float(coords[i,0]),
            "y_1":float(coords[i,1]),
            "z_1":float(coords[i,2]),
            "x_2":float(coords[i,0]),
            "y_2":float(coords[i,1]),
            "z_2":float(coords[i,2]),
            "x_3":float(coords[i,0]),
            "y_3":float(coords[i,1]),
            "z_3":float(coords[i,2]),
            "x_4":float(coords[i,0]),
            "y_4":float(coords[i,1]),
            "z_4":float(coords[i,2]),
            "x_5":float(coords[i,0]),
            "y_5":float(coords[i,1]),
            "z_5":float(coords[i,2]),
        })

submission=pd.DataFrame(rows)
submission.to_csv("submission.csv",index=False)

print("submission.csv saved")

submission.csv saved
